# Develop quick stuff

For temporary/quick stuff to be developed

In [1]:
import os
import sys

base_path = os.path.abspath("./..")
thesis_path = os.path.join(base_path, "thesis")
for p in [base_path, thesis_path]:
  if p not in sys.path:
      sys.path.append(p)

from IPython.display import display, Markdown
import torch


from experiments.acceptance_loop_cv_based_MNAR import THRESHOLD_BASIS, METRIC_CATEGORIES, EXPECTATION_TYPES, REAL_PERFORMANCE_TYPES
from berebasl.simulation.credit_data_simulation import CreditDataGenerator, GaussianMixture

In [2]:
os.chdir(base_path)
%run "thesis/helper_scripts/general.py"

In [ ]:
os.chdir(thesis_path)


grid_path = '../berebasl/data/simulations/mnar_grid_hyp_vec'
grid_biases = extract_biases_from_dirnames(grid_path)

all_sim_objs = extract_all_sim_objs_vec(grid_path, grid_biases)
all_corrs = torch.stack([sim_objs['corr_to_hidden'] for sim_objs in all_sim_objs.values()], dim=0)
assert torch.all(all_corrs[[0]] == all_corrs)

base_sim_bias = '0_0'
base_sim = all_sim_objs[base_sim_bias]
wished_corr = -0.4
base_sim_objs = slice_vec_sim(base_sim, wished_corr)

base_dgp_unsliced = base_sim["data_generator"]
dgp_base = base_sim_objs["data_generator"]

torch.Size([7, 306000])


In [6]:
from berebasl.simulation.credit_data_simulation import CreditData
credit_data: CreditData = base_sim["credit_data"]

In [ ]:
from berebasl.estimation.classifiers import BatchedLogistic

feats, lbls, mask_valid, gen_round = credit_data.accepts(include_gen_round=True, include_mask_valid=True)
G = credit_data.last_gen_round+1
Co, N, F = feats.shape

feats = feats.unsqueeze(1).expand(Co, G, N, F)
lbls = lbls.unsqueeze(1).expand(Co, G, N)

arange_G = torch.arange(G, device=gen_round.device, dtype=gen_round.dtype)

mask_gen_round_ok = gen_round.unsqueeze(1) <= arange_G.view(1, G, 1)

mask_valid_and_gen_round_ok = mask_valid.unsqueeze(1) & mask_gen_round_ok


batched_lr = BatchedLogistic(n_features=F-1, batch_shape = lbls.shape[:-1], device=feats.device, dtype=feats.dtype)
print("Starting training")
batched_lr.fit(feats[..., :-1], lbls, mask_valid_obs=mask_valid_and_gen_round_ok)

_LinAlgError: linalg.cholesky: (Batch element 0): The factorization could not be completed because the input is not positive-definite (the leading minor of order 1 is not positive-definite).